In [13]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import seaborn as sns



In [14]:
df = sns.load_dataset('tips')

In [12]:
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [13]:

df.columns

Index(['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size'], dtype='str')

In [15]:
# select feature and variable 
X = df.drop('tip', axis=1)
y = df['tip']

le = LabelEncoder()
X['sex'] = le.fit_transform(X['sex'])
X['smoker'] = le.fit_transform(X['smoker'])
X['day'] = le.fit_transform(X['day'])
X['time'] = le.fit_transform(X['time'])


x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=42)


In [16]:

models = {
    'LinearRegression': LinearRegression(),
    'SVR': SVR(),
    'DecisionTreeRegressor': DecisionTreeRegressor(),
    'RandomForestRegressor': RandomForestRegressor(),
    'KNeighborsRegressor': KNeighborsRegressor(),
    'GradientBoostingRegressor': GradientBoostingRegressor(),
    'XGBRegressor': XGBRegressor()
}

models_score = []

for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    metric = mean_absolute_error(y_test, y_pred)
    models_score.append((name, metric))
    
sorted_models = sorted(models_score, key=lambda x: x[1], reverse=False)

for name, score in sorted_models:
    print(f"Mean Absolute Error for {name} is {score:.2f}")

Mean Absolute Error for SVR is 0.57
Mean Absolute Error for LinearRegression is 0.67
Mean Absolute Error for XGBRegressor is 0.67
Mean Absolute Error for KNeighborsRegressor is 0.73
Mean Absolute Error for GradientBoostingRegressor is 0.73
Mean Absolute Error for RandomForestRegressor is 0.78
Mean Absolute Error for DecisionTreeRegressor is 0.94


In [17]:
models = {
    "LinearRegression": (LinearRegression(), {}),
    "SVR": (SVR(), {"kernel": ["rbf", "poly", "sigmoid"]}),
    "DecisionTreeRegressor": (
        DecisionTreeRegressor(),
        {
            "max_depth": [None, 5, 10],
        },
    ),
    "RandomForestRegressor": (
        RandomForestRegressor(),
        {
            "n_estimators": [10, 100],
        },
    ),
    "KNeighborsRegressor": (
        KNeighborsRegressor(),
        {
            "n_neighbors": list(range(3, 100, 2)),
        },
    ),
    "GradientBoostingRegressor": (
        GradientBoostingRegressor(),
        {"n_estimators": [10, 100]},
    ),
    "XGBRegressor": (
        XGBRegressor(),
        {
            "n_estimators": [10, 100],
        },
    ),
}

for name, (model, params) in models.items():
    pipeline = GridSearchCV(model, params, cv=5)

    pipeline.fit(x_train, y_train)
    
    y_pred = pipeline.predict(x_test)
    
    print(name, 'MSE: ', mean_squared_error(y_test, y_pred))
    print(name, 'R2: ', r2_score(y_test, y_pred))
    print(name, 'MAE: ', mean_absolute_error(y_test, y_pred))
    print('\n')

LinearRegression MSE:  0.6948129686287711
LinearRegression R2:  0.4441368826121931
LinearRegression MAE:  0.6703807496461157


SVR MSE:  1.460718141299992
SVR R2:  -0.1686013018011976
SVR MAE:  0.8935334948775431


DecisionTreeRegressor MSE:  0.8774153020453993
DecisionTreeRegressor R2:  0.298051667053291
DecisionTreeRegressor MAE:  0.7189481629481629


RandomForestRegressor MSE:  0.9864621291836747
RandomForestRegressor R2:  0.21081220548429302
RandomForestRegressor MAE:  0.7796755102040818


KNeighborsRegressor MSE:  0.6640950568462677
KNeighborsRegressor R2:  0.4687117753876745
KNeighborsRegressor MAE:  0.6203721488595437


GradientBoostingRegressor MSE:  0.8106801524004932
GradientBoostingRegressor R2:  0.35144101065487676
GradientBoostingRegressor MAE:  0.7657809818712309


XGBRegressor MSE:  0.6624107100882575
XGBRegressor R2:  0.4700592836840687
XGBRegressor MAE:  0.6549163442728472




In [ ]:
standardize_col = ["total_bill"]
preprocessor = ColumnTransformer(
    transformers=[("scaler", StandardScaler(), standardize_col)],
    remainder="passthrough",
)


models = {
    "LinearRegression": (LinearRegression(), {}),
    "SVR": (SVR(), {"kernel": ["rbf", "poly", "sigmoid"]}),
    "DecisionTreeRegressor": (
        DecisionTreeRegressor(),
        {
            "max_depth": [None, 5, 10],
        },
    ),
    "RandomForestRegressor": (
        RandomForestRegressor(),
        {
            "n_estimators": [10, 100],
        },
    ),
    "KNeighborsRegressor": (
        KNeighborsRegressor(),
        {
            "n_neighbors": list(range(3, 100, 2)),
        },
    ),
    "GradientBoostingRegressor": (
        GradientBoostingRegressor(),
        {"n_estimators": [10, 100]},
    ),
    "XGBRegressor": (
        XGBRegressor(),
        {
            "n_estimators": [10, 100],
        },
    ),
}

for name, (model, params) in models.items():

    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    grid_search = GridSearchCV(pipeline, params, cv=5)

    grid_search.fit(x_train, y_train)
    print(f"Best Parameter of {name} is ", grid_search.best_params_)
    y_pred = pipeline.predict(x_test)

    print(name, "MSE: ", mean_squared_error(y_test, y_pred))
    print(name, "R2: ", r2_score(y_test, y_pred))
    print(name, "MAE: ", mean_absolute_error(y_test, y_pred))
    print("\n")